In [22]:
from langchain_text_splitters import CharacterTextSplitter, TokenTextSplitter
from langchain_core.documents import Document

docs = [
    Document(page_content="this is document one. this has several sentences" * 3),
    Document(page_content="This is document tow. this has less sentences " )
]
print(type(docs))

splitter = CharacterTextSplitter(
    separator="", ##By default chuck is happened based on new line which is \n\n
    chunk_size=10,
    chunk_overlap=0
)

"""
Relation between separator(divides into tiny chunk) and chunk_size(basically it counts the character)
if separator chunk within chunk size, it will divide
if separator chunk is greater than chunk size

"""
chunks = splitter.split_documents(docs)

print(chunks)

print("Fixed chunk size : ",len(chunks))

for i, c in  enumerate(chunks):
    print(f'\nchunk{i} :  {c.page_content}')

Created a chunk of size 13, which is longer than the specified 10
Created a chunk of size 13, which is longer than the specified 10


<class 'list'>
[Document(metadata={}, page_content='this is'), Document(metadata={}, page_content='document'), Document(metadata={}, page_content='one. this'), Document(metadata={}, page_content='has'), Document(metadata={}, page_content='several'), Document(metadata={}, page_content='sentencesthis'), Document(metadata={}, page_content='is'), Document(metadata={}, page_content='document'), Document(metadata={}, page_content='one. this'), Document(metadata={}, page_content='has'), Document(metadata={}, page_content='several'), Document(metadata={}, page_content='sentencesthis'), Document(metadata={}, page_content='is'), Document(metadata={}, page_content='document'), Document(metadata={}, page_content='one. this'), Document(metadata={}, page_content='has'), Document(metadata={}, page_content='several'), Document(metadata={}, page_content='sentences'), Document(metadata={}, page_content='This is'), Document(metadata={}, page_content='document'), Document(metadata={}, page_content='tow. t

In [35]:
#Overlap Scenario
text=("Sentence one. Sentence two. Sentence three. ")*10

docs1 = [
    Document(page_content=text)
]
chunk_overlap=3
#next chunk will include first all the character from the last, overlap is defined
splitter = CharacterTextSplitter(
    ##By default chuck is happened based on new line which is \n\n
     separator="",
    chunk_size=15,
    chunk_overlap=chunk_overlap

)


chunks = splitter.split_documents(docs1)

print(chunks)

print("Fixed chunk size : ",len(chunks))

for i, c in  enumerate(chunks):
    print(f'\nchunk{i} :{c.page_content}:length {len(c.page_content)}' )

for i in range(len(chunks)-1):
    a=chunks[i].page_content
    b=  chunks[i+1].page_content

    overlap_a=a[-chunk_overlap:]
    overlap_b=b[:chunk_overlap]

    print(f'\n Ovrelap between chunk {i} and {i+1}')
    print("From end of chunk ", i, " : ",repr(overlap_a))
    print("From start of chunk ", i+1, " : ",repr(overlap_b))
    print("Identical :", overlap_a==overlap_b)

[Document(metadata={}, page_content='Sentence one. S'), Document(metadata={}, page_content='. Sentence two.'), Document(metadata={}, page_content='wo. Sentence th'), Document(metadata={}, page_content='three. Sentenc'), Document(metadata={}, page_content='ence one. Sente'), Document(metadata={}, page_content='ntence two. Sen'), Document(metadata={}, page_content='Sentence three.'), Document(metadata={}, page_content='ee. Sentence on'), Document(metadata={}, page_content='one. Sentence'), Document(metadata={}, page_content='ce two. Sentenc'), Document(metadata={}, page_content='ence three. Sen'), Document(metadata={}, page_content='Sentence one. S'), Document(metadata={}, page_content='. Sentence two.'), Document(metadata={}, page_content='wo. Sentence th'), Document(metadata={}, page_content='three. Sentenc'), Document(metadata={}, page_content='ence one. Sente'), Document(metadata={}, page_content='ntence two. Sen'), Document(metadata={}, page_content='Sentence three.'), Document(meta

In [1]:
import nltk
from nltk.tokenize import sent_tokenize

# Download once
nltk.download("punkt")

# -----------------------------
# Recursive Chunking Function
# -----------------------------
def recursive_chunk_text(
    text,
    max_words=50,
    separators=["\n\n", "\n", ".", " ", ""]
):
    """
    Recursively splits text using decreasingly smaller separators
    until each chunk is <= max_words
    """

    def split_text(text, sep):
        if sep == "":
            return list(text)
        return text.split(sep)

    words = text.split()
    if len(words) <= max_words:
        return [text.strip()]

    for sep in separators:
        parts = split_text(text, sep)
        chunks = []

        current_chunk = ""
        for part in parts:
            candidate = current_chunk + (sep if current_chunk else "") + part
            if len(candidate.split()) <= max_words:
                current_chunk = candidate
            else:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                current_chunk = part

        if current_chunk:
            chunks.append(current_chunk.strip())

        # If split helped, recurse further if needed
        if len(chunks) > 1:
            final_chunks = []
            for chunk in chunks:
                if len(chunk.split()) > max_words:
                    final_chunks.extend(
                        recursive_chunk_text(chunk, max_words, separators[1:])
                    )
                else:
                    final_chunks.append(chunk)
            return final_chunks

    return [text.strip()]

long_text = """
Artificial Intelligence is transforming modern technology.
It is widely used in healthcare for diagnosis and treatment.

Machine learning is a subset of AI that focuses on learning from data.
Deep learning uses neural networks for complex tasks.

Climate change is a major global challenge.
Renewable energy sources like solar and wind are critical.
"""

chunks = recursive_chunk_text(long_text, max_words=20)

for i, chunk in enumerate(chunks, 1):
    print(f"\n--- Chunk {i} ---")
    print(chunk)


--- Chunk 1 ---
Artificial Intelligence is transforming modern technology.
It is widely used in healthcare for diagnosis and treatment.

--- Chunk 2 ---
Machine learning is a subset of AI that focuses on learning from data.

--- Chunk 3 ---
Deep learning uses neural networks for complex tasks.

--- Chunk 4 ---
Climate change is a major global challenge.
Renewable energy sources like solar and wind are critical.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rshandil\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
